# HR recruitment graph (Part 2)

Same example as `00_thinking_in_langgraph.md`: **extract_skills → score_candidate → interview | reject**.

In [ ]:
!uv add langgraph

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class RecruitmentState(TypedDict):
    candidate_name: str
    resume_text: str
    job_title: str
    required_skills: list[str]
    skills_found: list[str] | None
    match_score: float | None
    qualified: bool | None
    outcome: Literal["interview", "rejected"] | None
    message: str | None

In [ ]:
KEYWORDS = {"python", "sql", "fastapi", "docker", "pytest", "langchain"}
PASS_THRESHOLD = 0.6


def extract_skills(state: RecruitmentState) -> dict:
    text = state["resume_text"].lower()
    return {"skills_found": [kw for kw in KEYWORDS if kw in text]}


def score_candidate(state: RecruitmentState) -> dict:
    found = set(state.get("skills_found") or [])
    required = set(state["required_skills"])
    score = len(found & required) / len(required) if required else 0.0
    return {"match_score": round(score, 2), "qualified": score >= PASS_THRESHOLD}


def schedule_interview(state: RecruitmentState) -> dict:
    return {
        "outcome": "interview",
        "message": f"Interview booked for {state['candidate_name']} ({state['job_title']}).",
    }


def send_rejection(state: RecruitmentState) -> dict:
    return {
        "outcome": "rejected",
        "message": (
            f"Dear {state['candidate_name']}, match {state['match_score']:.0%} — not proceeding."
        ),
    }


def route_decision(state: RecruitmentState) -> str:
    return "interview" if state["qualified"] else "reject"


graph = (
    StateGraph(RecruitmentState)
    .add_node("extract_skills", extract_skills)
    .add_node("score_candidate", score_candidate)
    .add_node("schedule_interview", schedule_interview)
    .add_node("send_rejection", send_rejection)
    .add_edge(START, "extract_skills")
    .add_edge("extract_skills", "score_candidate")
    .add_conditional_edges(
        "score_candidate",
        route_decision,
        {"interview": "schedule_interview", "reject": "send_rejection"},
    )
    .add_edge("schedule_interview", END)
    .add_edge("send_rejection", END)
    .compile()
)

In [ ]:
from IPython.display import Markdown

Markdown(f"```mermaid\n{graph.get_graph().draw_mermaid()}\n```")

In [ ]:
JOB = {
    "job_title": "Python Developer",
    "required_skills": ["python", "sql", "fastapi"],
}

CVS: list[dict] = [
    {
        "candidate_name": "Anna Kowalski",
        "resume_text": """
        Anna Kowalski — Senior Backend Engineer
        5 years building APIs with Python and FastAPI.
        PostgreSQL, SQL queries, Docker deployments, pytest suites.
        """,
    },
    {
        "candidate_name": "Mark Nowak",
        "resume_text": """
        Mark Nowak — Marketing Coordinator
        Campaign planning, social media, Excel reports, PowerPoint decks.
        No software development experience.
        """,
    },
    {
        "candidate_name": "Piotr Zielinski",
        "resume_text": """
        Piotr Zielinski — Junior Developer
        Completed a Python bootcamp. Built small scripts in Python.
        Wants to learn databases and API frameworks next.
        """,
    },
    {
        "candidate_name": "Kasia Lewandowska",
        "resume_text": """
        Kasia Lewandowska — Full-stack Developer
        Python microservices, FastAPI, raw SQL + ORMs, langchain prototypes.
        pytest, Docker, CI pipelines.
        """,
    },
]

In [ ]:
def empty_state(cv: dict) -> RecruitmentState:
    return {
        **cv,
        **JOB,
        "skills_found": None,
        "match_score": None,
        "qualified": None,
        "outcome": None,
        "message": None,
    }


for cv in CVS:
    result = graph.invoke(empty_state(cv))
    edge = (
        "schedule_interview" if result["outcome"] == "interview" else "send_rejection"
    )
    print(
        f"{result['candidate_name']:20} score={result['match_score']:.0%}  edge -> {edge}"
    )
    print(f"  skills: {result['skills_found']}")
    print(f"  {result['message']}\n")